# NB-06 · Training Dynamics Diagnostics

**Checks covered**
1. Single-batch overfit: can the model memorize 1 patch in ~50 steps?
2. Gradient norms per parameter group (encoder / decoder / head)
3. Gradient explosion / vanishing detection across layers
4. LR schedule shape: warmup + cosine annealing curve
5. EMA weight divergence from live weights over training
6. Mini train/val run (5 epochs, few patches): loss decreases monotonically?
7. Weight update magnitude vs. parameter magnitude (update ratio)

In [ ]:
import sys, json, copy
from pathlib import Path
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

REPO_ROOT    = Path("../..").resolve()
DATASET_PATH = Path("/ste/rnd/User/vice_vi/Dataset/clean_dataset")
PARAMS_PATH  = DATASET_PATH / "params" / "params_sig_k5" / "parameters_sig_k5.npy"
DATA_DIR     = DATASET_PATH / "data"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from torch.utils.data import DataLoader, Subset
from tools.logger                    import Logger
from tools.tracker                   import Tracker
from tools.crop_region               import CropRegion
from configuration.dataset_config    import InputConfig, OutputConfig, PatchConfiguration, Representation
from configuration.training_config   import GaussianConfig, LossConfig
from pipelines.dataset_pipeline.patch   import Patcher
from pipelines.dataset_pipeline.load    import PatchDataset
from pipelines.dataset_pipeline.normalize import Stats, StatsComputer, Normalizer
from pipelines.training_pipeline.loss   import Loss
from models import build_model, UNetConfig

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GAUSSIANS = 5
PATCH_SIZE  = (64, 64)
STRIDE      = 32

logger  = Logger(name="nb06", logdir="/tmp", run_name="nb06", verbose=False)
tracker = Tracker(writer=None)

with open(DATA_DIR / "dataset.json") as f:
    layout = json.load(f)
gc_crop = CropRegion(*layout["global_crop"])

gaussian_cfg = GaussianConfig.from_dataset(DATASET_PATH, n_gaussians=N_GAUSSIANS)
x_axis       = torch.linspace(gaussian_cfg.x_min, gaussian_cfg.x_max, 128).to(DEVICE)

print(f"Device       : {DEVICE}")
print(f"x_axis range : [{x_axis.min():.2f}, {x_axis.max():.2f}]")

## 0 · Build a tiny dataset (50 patches from train split)

In [ ]:
primary        = np.load(next(DATA_DIR.glob("primary_*.npy")),        mmap_mode='r')
secondaries    = np.load(next(DATA_DIR.glob("secondaries_*.npy")),    mmap_mode='r')
interferograms = np.load(next(DATA_DIR.glob("interferograms_*.npy")), mmap_mode='r')
params_full    = np.load(PARAMS_PATH,                                 mmap_mode='r')

az0, az1 = 1000, 1512    # small window for speed
rg0, rg1 = gc_crop.range_start, gc_crop.range_end

inputs_crop = np.concatenate([
    primary     [np.newaxis, az0:az1, rg0:rg1],
    secondaries [:, az0:az1, rg0:rg1],
    interferograms[:, az0:az1, rg0:rg1],
], axis=0).astype(np.complex64)
params_crop = params_full[:, az0:az1, rg0:rg1].astype(np.float32)

input_config  = InputConfig(
    use_primary=True,  primary_representation=Representation.MAG_ONLY,
    use_secondaries=True, secondaries_representation=Representation.MAG_ONLY,
    use_interferograms=True, interferograms_representation=Representation.ANGLE_ONLY,
)
output_config = OutputConfig()
patcher       = Patcher.build((inputs_crop.shape[1], inputs_crop.shape[2]), PATCH_SIZE, STRIDE)

ds_raw = PatchDataset(
    inputs=inputs_crop, gt_parameters=params_crop, grid=patcher,
    input_config=input_config, output_config=output_config,
    split_name="train", n_gaussians=N_GAUSSIANS, norm_stats=None,
)

# Fit normalization on these patches
n_slaves = secondaries.shape[0]
in_stats = StatsComputer.compute_input_stats(
    ds_raw, logger, input_config, n_slaves, max_samples=500, num_workers=0, batch_size=128,
)
out_stats = StatsComputer.compute_output_stats(PARAMS_PATH, N_GAUSSIANS, output_config, logger=logger)
norm_stats = Stats(input_stats=in_stats.input_stats, output_stats=out_stats)

ds = PatchDataset(
    inputs=inputs_crop, gt_parameters=params_crop, grid=patcher,
    input_config=input_config, output_config=output_config,
    split_name="train", n_gaussians=N_GAUSSIANS, norm_stats=norm_stats,
)

N_MINI = 50
rng    = np.random.default_rng(0)
idxs   = rng.choice(len(ds), N_MINI, replace=False).tolist()
ds_mini = Subset(ds, idxs)

x0, y0 = ds[0]
IN_CH  = x0.shape[0]
OUT_CH = y0.shape[0]
print(f"Patches : {N_MINI}  |  Input ch: {IN_CH}  |  Output ch: {OUT_CH}")

## 1 · Build a small U-Net

In [ ]:
from tools.normalizer_wrapper import NormalizerWrapper

model_cfg = UNetConfig()
model     = build_model("unet", model_cfg, in_channels=IN_CH, out_channels=OUT_CH)
model     = model.to(DEVICE)

norm_wrapper = NormalizerWrapper(norm_stats)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model params : {total_params:,}")

## 2 · LR schedule shape (warmup + cosine annealing)

In [ ]:
from pipelines.training_pipeline.warmup    import Warmup
from pipelines.training_pipeline.scheduler import Scheduler
from configuration.training_config import (
    TrainerConfig, TrainingConfigInner, IOConfig, WarmupConfig, SchedulerConfig,
    EarlyStoppingConfig, EMAConfig, OptimizerConfig, OverfitConfig,
)

EPOCHS       = 30
WARMUP_STEPS = 50

cfg = TrainerConfig(
    gaussian  = gaussian_cfg,
    io        = IOConfig(logdir="/tmp"),
    training  = TrainingConfigInner(device="cpu", epochs=EPOCHS, verbose=False),
    warmup    = WarmupConfig(warmup_steps=WARMUP_STEPS, warmup_start_factor=0.1, warmup_enabled=True),
    scheduler = SchedulerConfig(type="cosine_annealing", epochs=EPOCHS, eta_min=1e-6),
    ema       = EMAConfig(use_ema=False),
    optimizer = OptimizerConfig(betas=(0.9,0.999), eps=1e-8),
    early_stopping = EarlyStoppingConfig(patience=100),
    overfit        = OverfitConfig(),
    loss           = LossConfig(use_param_l1=True, weight_param_l1=1.0),
)

# Simulate LR trajectory
base_lr   = 3e-4
optimizer_sim = optim.AdamW([{'params': model.parameters(), 'lr': base_lr, 'name': 'all'}])
warmup_sim    = Warmup(cfg, logger, tracker)
sched_sim     = Scheduler([base_lr], warmup_sim, cfg, logger, tracker)

STEPS_PER_EPOCH = 10
lr_trace = []

for epoch in range(EPOCHS):
    for step in range(STEPS_PER_EPOCH):
        warmup_sim.step()
        # get current LR (warmup overrides)
        lrs = warmup_sim.get_lrs([base_lr])
        lr_trace.append(lrs[0])
    sched_sim.step(epoch, metric=0.5)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(lr_trace, color="steelblue")
ax.axvline(WARMUP_STEPS, color='red', linestyle='--', label=f'Warmup end (step {WARMUP_STEPS})')
ax.set_xlabel("Step")
ax.set_ylabel("LR")
ax.set_title("LR schedule: warmup + cosine annealing")
ax.legend()
plt.tight_layout()
plt.show()

print(f"LR at step 0          : {lr_trace[0]:.2e}")
print(f"LR at step {WARMUP_STEPS}         : {lr_trace[WARMUP_STEPS]:.2e}")
print(f"LR at end             : {lr_trace[-1]:.2e}")

## 3 · Single-batch overfit test

In [ ]:
OVERFIT_STEPS = 200
OVERFIT_LR    = 1e-3

loss_cfg    = LossConfig(use_param_l1=True, weight_param_l1=1.0, param_match="sort_gt_by_mu")
criterion   = Loss(x_axis, logger, tracker, gaussian_cfg, loss_cfg, norm_stats=norm_wrapper)

# Single batch
x_batch, y_batch = ds_mini[0]
x_b = torch.from_numpy(x_batch[None]).to(DEVICE)
y_b = torch.from_numpy(y_batch[None]).to(DEVICE)

# Fresh model copy for this test
m_overfit = copy.deepcopy(model)
opt_of    = optim.Adam(m_overfit.parameters(), lr=OVERFIT_LR)

overfit_losses = []
m_overfit.train()
for step in range(OVERFIT_STEPS):
    opt_of.zero_grad()
    pred = m_overfit(x_b)
    out  = criterion(pred, y_b)
    loss = out["total_loss"]
    loss.backward()
    opt_of.step()
    overfit_losses.append(loss.item())

init_loss  = overfit_losses[0]
final_loss = overfit_losses[-1]
ratio      = final_loss / (init_loss + 1e-12)

fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogy(overfit_losses, color="darkorange")
ax.set_xlabel("Step")
ax.set_ylabel("Loss (log scale)")
ax.set_title("Single-batch overfit")
plt.tight_layout()
plt.show()

print(f"Initial loss : {init_loss:.4e}")
print(f"Final loss   : {final_loss:.4e}")
print(f"Reduction    : {ratio:.4f}x")
if ratio < 0.1:
    print("✓ Model can memorize a single batch")
elif ratio < 0.5:
    print("~ Partial memorization – may need more steps or higher LR")
else:
    print("⚠ Model is NOT memorizing – check architecture, loss, or input pipeline")

## 4 · Gradient norms per layer

In [ ]:
m_gnorm = copy.deepcopy(model).train().to(DEVICE)
opt_gn  = optim.Adam(m_gnorm.parameters(), lr=1e-3)

opt_gn.zero_grad()
pred = m_gnorm(x_b)
out  = criterion(pred, y_b)
out["total_loss"].backward()

layer_norms  = []
layer_names  = []

for name, param in m_gnorm.named_parameters():
    if param.grad is not None:
        gnorm = param.grad.norm().item()
        pnorm = param.data.norm().item()
        layer_norms.append((name, gnorm, pnorm, gnorm / (pnorm + 1e-12)))

# Summary by group
print(f"  {'Layer (truncated)':<45s}  {'‖grad‖':>10s}  {'‖param‖':>10s}  {'ratio':>10s}  {'Status'}")
print("  " + "-" * 90)
for name, gn, pn, ratio in layer_norms:
    trunc = name[-43:] if len(name) > 43 else name
    if ratio > 1.0:
        status = "⚠ EXPLODING?"
    elif ratio < 1e-6:
        status = "⚠ VANISHING?"
    else:
        status = "✓"
    print(f"  {trunc:<45s}  {gn:>10.3e}  {pn:>10.3e}  {ratio:>10.3e}  {status}")

## 5 · Mini training run (5 epochs, 50 patches)

In [ ]:
MINI_EPOCHS = 10
MINI_BS     = 8

# Split 50 patches into 40 train / 10 val
all_idxs  = list(range(N_MINI))
tr_idxs   = all_idxs[:40]
va_idxs   = all_idxs[40:]
ds_tr     = Subset(ds, [idxs[i] for i in tr_idxs])
ds_va     = Subset(ds, [idxs[i] for i in va_idxs])

dl_tr = DataLoader(ds_tr, batch_size=MINI_BS, shuffle=True,  num_workers=0)
dl_va = DataLoader(ds_va, batch_size=MINI_BS, shuffle=False, num_workers=0)

m_mini   = copy.deepcopy(model).to(DEVICE)
opt_mini = optim.AdamW(m_mini.parameters(), lr=3e-4, weight_decay=1e-4)

train_losses, val_losses = [], []

for epoch in range(MINI_EPOCHS):
    # Train
    m_mini.train()
    ep_loss = 0.0
    for x_b_, y_b_ in dl_tr:
        x_b_ = x_b_.to(DEVICE); y_b_ = y_b_.to(DEVICE)
        opt_mini.zero_grad()
        pred = m_mini(x_b_)
        out  = criterion(pred, y_b_)
        out["total_loss"].backward()
        opt_mini.step()
        ep_loss += out["total_loss"].item()
    train_losses.append(ep_loss / max(1, len(dl_tr)))

    # Val
    m_mini.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_b_, y_b_ in dl_va:
            x_b_ = x_b_.to(DEVICE); y_b_ = y_b_.to(DEVICE)
            pred = m_mini(x_b_)
            out  = criterion(pred, y_b_)
            val_loss += out["total_loss"].item()
    val_losses.append(val_loss / max(1, len(dl_va)))

    print(f"  Epoch {epoch+1:>3d}/{MINI_EPOCHS}: train={train_losses[-1]:.4e}  val={val_losses[-1]:.4e}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(range(1, MINI_EPOCHS+1), train_losses, label="train", color="steelblue",   marker="o")
ax.semilogy(range(1, MINI_EPOCHS+1), val_losses,   label="val",   color="darkorange", marker="s")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Mini training run")
ax.legend()
plt.tight_layout()
plt.show()

monotone = all(train_losses[i] >= train_losses[i+1] for i in range(len(train_losses)-1))
print(f"Train loss monotonically decreasing? {'✓ yes' if monotone else '~ not strictly (may be OK with small dataset)'}")
if train_losses[-1] < train_losses[0] * 0.9:
    print("✓ Train loss decreased by >10%")
else:
    print("⚠  Train loss barely moved – check LR, loss config, or data pipeline")

## 6 · EMA weight divergence from live weights

In [ ]:
from pipelines.training_pipeline.ema import EMA

cfg_ema = TrainerConfig(
    gaussian  = gaussian_cfg,
    io        = IOConfig(logdir="/tmp"),
    training  = TrainingConfigInner(device="cpu", epochs=50, verbose=False),
    warmup    = WarmupConfig(warmup_enabled=False),
    scheduler = SchedulerConfig(type="cosine_annealing", epochs=50, eta_min=1e-6),
    ema       = EMAConfig(use_ema=True, ema_decay=0.99),
    optimizer = OptimizerConfig(),
    early_stopping = EarlyStoppingConfig(patience=100),
    overfit        = OverfitConfig(),
    loss           = LossConfig(use_param_l1=True, weight_param_l1=1.0),
)

m_ema  = copy.deepcopy(model).to(DEVICE)
ema    = EMA(cfg_ema, logger, tracker)
ema.init(m_ema)
opt_em = optim.Adam(m_ema.parameters(), lr=1e-3)

divergences = []
EMA_STEPS   = 100

for step in range(EMA_STEPS):
    x_b_, y_b_ = ds_mini[step % N_MINI]
    x_b_ = torch.from_numpy(x_b_[None]).to(DEVICE)
    y_b_ = torch.from_numpy(y_b_[None]).to(DEVICE)

    opt_em.zero_grad()
    pred = m_ema(x_b_)
    out  = criterion(pred, y_b_)
    out["total_loss"].backward()
    opt_em.step()
    ema.update(m_ema, step=step)

    # Measure divergence: mean abs diff between live and EMA weights
    with torch.no_grad():
        diffs = []
        for (name, live_p), ema_p in zip(m_ema.named_parameters(), ema._shadow.values()):
            diffs.append((live_p - ema_p).abs().mean().item())
        divergences.append(np.mean(diffs))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(divergences, color="purple")
ax.set_xlabel("Step")
ax.set_ylabel("Mean |live - EMA|")
ax.set_title("EMA weight divergence from live weights")
plt.tight_layout()
plt.show()
print(f"Final EMA divergence : {divergences[-1]:.4e}  (should stabilize, not explode)")